# Movie Recommendation for Streaming Platform
## Notebook 1 — Exploratory Data Analysis (EDA)

**Objective:** Understand the MovieLens dataset — its structure, size, users, movies, and rating behaviour — before building any model.

**Files used:**
- `ratings.DAT` — user ratings for movies
- `movies.DAT` — movie titles and genres
- `users.DAT` — user demographic info

In [1]:
import pandas as pd
import numpy as np

## Step 1 — Load the Three Datasets

In [2]:
path        = r'C:\Movie Recommendation project\data\ratings.DAT'
movies_path = r'C:\Movie Recommendation project\data\movies.DAT'
users_path  = r'C:\Movie Recommendation project\data\users.DAT'

ratings = pd.read_csv(path,        sep='::', engine='python',
                      names=['user_id','movie_id','rating','timestamp'], encoding='iso-8859-1')
movies  = pd.read_csv(movies_path, sep='::', engine='python',
                      names=['movie_id','title','genres'],               encoding='iso-8859-1')
users   = pd.read_csv(users_path,  sep='::', engine='python',
                      names=['user_id','gender','age','occupation','zip_code'], encoding='iso-8859-1')

print('Ratings shape :', ratings.shape)
print('Movies shape  :', movies.shape)
print('Users shape   :', users.shape)

Ratings shape : (1000209, 4)
Movies shape  : (3883, 3)
Users shape   : (6040, 5)


## Step 2 — Preview Each Dataset

In [3]:
print('=== Ratings ===')
print(ratings.head())
print('\n=== Movies ===')
print(movies.head())
print('\n=== Users ===')
print(users.head())

=== Ratings ===
   user_id  movie_id  rating  timestamp
0        1      1193       5  978300760
1        1       661       3  978302109
2        1       914       3  978301968
3        1      3408       4  978300275
4        1      2355       5  978824291

=== Movies ===
   movie_id                               title                        genres
0         1                    Toy Story (1995)   Animation|Children's|Comedy
1         2                      Jumanji (1995)  Adventure|Children's|Fantasy
2         3             Grumpier Old Men (1995)                Comedy|Romance
3         4            Waiting to Exhale (1995)                  Comedy|Drama
4         5  Father of the Bride Part II (1995)                        Comedy

=== Users ===
   user_id gender  age  occupation zip_code
0        1      F    1          10    48067
1        2      M   56          16    70072
2        3      M   25          15    55117
3        4      M   45           7    02460
4        5      M   25   

## Step 3 — Map Occupation Codes to Readable Names

In [4]:
occupation_dict = {
    0:'Other', 1:'Academic/Educator', 2:'Artist', 3:'Clerical/Admin',
    4:'College/Grad Student', 5:'Customer Service', 6:'Doctor/Health Care',
    7:'Executive/Managerial', 8:'Farmer', 9:'Homemaker', 10:'K-12 Student',
    11:'Lawyer', 12:'Programmer', 13:'Retired', 14:'Sales/Marketing',
    15:'Scientist', 16:'Self-Employed', 17:'Technician/Engineer',
    18:'Tradesman/Craftsman', 19:'Unemployed', 20:'Writer'
}
users['occupation_name'] = users['occupation'].map(occupation_dict)
users[['user_id','occupation','occupation_name']].head()

,user_id,occupation,occupation_name
0,1,10,K-12 Student
1,2,16,Self-Employed
2,3,15,Scientist
3,4,7,Executive/Managerial
4,5,20,Writer


## Step 4 — Merge All Three Datasets into One

In [5]:
data = pd.merge(ratings, movies, on='movie_id')
data = pd.merge(data,    users,  on='user_id')

print('Merged data shape:', data.shape)
print(data.head())

Merged data shape: (1000209, 11)
   user_id  movie_id  rating  timestamp  \
0        1      1193       5  978300760   
1        1       661       3  978302109   
2        1       914       3  978301968   
3        1      3408       4  978300275   
4        1      2355       5  978824291   

                                    title                        genres  \
0  One Flew Over the Cuckoo's Nest (1975)                         Drama   
1        James and the Giant Peach (1996)  Animation|Children's|Musical   
2                     My Fair Lady (1964)               Musical|Romance   
3                  Erin Brockovich (2000)                         Drama   
4                    Bug's Life, A (1998)   Animation|Children's|Comedy   

  gender  age  occupation zip_code occupation_name  
0      F    1          10    48067    K-12 Student  
1      F    1          10    48067    K-12 Student  
2      F    1          10    48067    K-12 Student  
3      F    1          10    48067    K-12 St

## Step 5 — Basic Data Quality Check

In [6]:
print('=== Missing Values ===')
print(data.isnull().sum())

print('\n=== Data Types ===')
print(data.dtypes)

print('\n=== Rating Range ===')
print(f"Min rating: {data['rating'].min()}   Max rating: {data['rating'].max()}")

=== Missing Values ===
user_id            0
movie_id           0
rating             0
timestamp          0
title              0
genres             0
gender             0
age                0
occupation         0
zip_code           0
occupation_name    0
dtype: int64

=== Data Types ===
user_id             int64
movie_id            int64
rating              int64
timestamp           int64
title              object
genres             object
gender             object
age                 int64
occupation          int64
zip_code           object
occupation_name    object
dtype: object

=== Rating Range ===
Min rating: 1   Max rating: 5


## Step 6 — Key Dataset Statistics

In [7]:
print(f"Total ratings        : {len(data):,}")
print(f"Unique users         : {data['user_id'].nunique():,}")
print(f"Unique movies        : {data['title'].nunique():,}")
print(f"Avg ratings per user : {data.groupby('user_id')['rating'].count().mean():.1f}")
print(f"Avg ratings per movie: {data.groupby('title')['rating'].count().mean():.1f}")
print(f"Overall avg rating   : {data['rating'].mean():.2f}")

Total ratings        : 1,000,209
Unique users         : 6,040
Unique movies        : 3,706
Avg ratings per user : 165.6
Avg ratings per movie: 269.9
Overall avg rating   : 3.58


## Step 7 — Gender and Age Distribution

In [8]:
print('=== Gender Distribution ===')
print(users['gender'].value_counts())

print('\n=== Age Group Distribution ===')
age_map = {1:'Under 18', 18:'18-24', 25:'25-34', 35:'35-44', 45:'45-49', 50:'50-55', 56:'56+'}
users['age_group'] = users['age'].map(age_map)
print(users['age_group'].value_counts())

=== Gender Distribution ===
gender
M    4331
F    1709
Name: count, dtype: int64

=== Age Group Distribution ===
age_group
25-34       2096
35-44       1193
18-24       1103
45-49        550
50-55        496
56+          380
Under 18     222
Name: count, dtype: int64


## Step 8 — Top Occupations Among Users

In [9]:
print('=== Top 10 Occupations ===')
print(users['occupation_name'].value_counts().head(10))

=== Top 10 Occupations ===
occupation_name
College/Grad Student    759
Other                   711
Executive/Managerial    679
Academic/Educator       528
Technician/Engineer     502
Programmer              388
Sales/Marketing         302
Writer                  281
Artist                  267
Self-Employed           241
Name: count, dtype: int64


## Step 9 — Save Merged Data for Use in Other Notebooks

In [10]:
data.to_csv(r'C:\Movie Recommendation project\data\merged_data.csv', index=False)
print('Saved merged_data.csv')

Saved merged_data.csv


## EDA Summary

| Observation | Finding |
|---|---|
| Dataset size | ~1 million ratings, 6040 users, 3706 movies |
| Missing values | None (clean dataset) |
| Rating scale | 1 to 5 |
| Dominant gender | Male users |
| Largest age group | 25–34 |
| Largest occupation | College/Grad Student |

The data is clean and ready for visualisation and modelling.